### Products Processing

#### Load Products from raw to Bronze


In [0]:
df_products_raw = spark.read \
    .format("csv") \
    .option("header", "true") \
    .load("/Volumes/ecommerce/raw/raw_aws_data/products/")

df_products_raw.printSchema()

df_products_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce.bronze.brz_products")

In [0]:
from pyspark.sql.functions import col 
df_products = spark.read.table("ecommerce.bronze.brz_products")
print(f"Products CSV has total {len(df_products.columns)} columns")

print("Unique Products ", df_products.select(col("product_id")).distinct().count()) # checking if we have all rows unique 
print("Total Rows", df_products.count())

df_products.show(5)

### Lets remove 'g' from weight columns and convert it into IntegerType.

In [0]:
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.functions import col,regexp_replace
df_products = df_products.withColumn("weight_grams", regexp_replace(col("weight_grams"),"g", "").cast(IntegerType()))\
                         .withColumn("length_cm", regexp_replace(col("length_cm"), ",", ".").cast(DoubleType()))


Cat and Brand Code make them UPCASE 

In [0]:
from pyspark.sql.functions import upper
df_products = df_products.withColumn("category_code", upper(col("category_code"))) \
                .withColumn("brand_code", upper(col("brand_code")))

df_products.show(5)

#### Using When which is Case version of SQL in Pyspark.

In [0]:
spark.sql("select  distinct material from ecommerce.bronze.brz_products").show()

In [0]:
from pyspark.sql.functions import when, col
df_products = df_products.withColumn(
     "material",
    when(col("material") == 'Coton', 'Cotton')
    .when(col("material") == 'Ruber', 'Rubber')
    .when(col("material") == 'Alumium', 'Aluminium')
    .otherwise(col("material"))
)
df_products.show(4)

In [0]:
from pyspark.sql.functions import abs , lit
df_products = df_products.withColumn("rating_count", 
                                     when(col("rating_count").isNotNull(), abs(col("rating_count")))
                                     .otherwise(lit(0))
                                    )

### Write as External delta Table.


In [0]:
df_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overWriteSchema", True) \
    .save("s3://sj-dbr-demo-proj/silver_data/slv_products")

spark.sql("""
          create table if not exists ecommerce.silver.slv_products
          using delta 
           location 's3://sj-dbr-demo-proj/silver_data/slv_products'
           """)


### lets test if table update affects underlaying filesi s3 ? 

In [0]:
spark.sql("select * from ecommerce.silver.slv_products where product_id = '2000000000015'").show()

In [0]:
spark.sql("""
          update  ecommerce.silver.slv_products set color = 'Rainbow' where product_id = '2000000000015'
          """)

In [0]:
dbutils.notebook.exit("SUCCESS")

### It changes Delta files on S3 